In [1]:
import duckdb

con = duckdb.connect()
con.execute("CREATE VIEW orders AS SELECT * FROM read_csv_auto('data/dataco_clean.csv')")

print(con.execute("SELECT COUNT(*) FROM orders").fetchall())

[(180519,)]


In [2]:
query = """
SELECT "Order Region" AS region,
       COUNT(*) AS items,
       ROUND(AVG("Late_delivery_risk") * 100, 1) AS late_pct
FROM orders
WHERE "Delivery Status" <> 'Shipping canceled'
GROUP BY region
ORDER BY late_pct DESC
"""
print(con.execute(query).df())

             region  items  late_pct
0    Central Africa   1616      60.1
1    Western Europe  25867      58.5
2    South of  USA    3855      58.5
3        South Asia   7455      58.4
4       East of USA   6617      58.2
5       East Africa   1781      58.2
6    Southeast Asia   9136      58.0
7         West Asia   5746      57.8
8    Eastern Europe   3785      57.6
9        US Center    5653      57.5
10    South America  14184      57.2
11  Central America  27174      57.1
12     North Africa   3086      57.1
13     West of USA    7595      56.8
14  Southern Europe   9030      56.8
15     Eastern Asia   6973      56.7
16     Central Asia    542      56.5
17  Northern Europe   9408      56.3
18          Oceania   9733      56.3
19  Southern Africa   1100      56.1
20        Caribbean   7951      55.5
21      West Africa   3571      54.7
22           Canada    907      51.6


In [3]:
query = """
SELECT "Late_delivery_risk" AS late,
       COUNT(*) AS items,
       ROUND(AVG("Order Profit Per Order"), 2) AS avg_profit,
       ROUND(AVG(CASE WHEN "Order Profit Per Order" < 0 THEN 1 ELSE 0 END) * 100, 1) AS loss_pct
FROM orders
WHERE "Delivery Status" <> 'Shipping canceled'
GROUP BY late
ORDER BY late
"""
print(con.execute(query).df())

   late  items  avg_profit  loss_pct
0     0  73788       22.58      18.6
1     1  98977       21.62      18.7


In [4]:
query = """
SELECT ROUND("Order Item Discount Rate" * 100) AS discount_pct,
       COUNT(*) AS items,
       ROUND(AVG("Order Profit Per Order"), 2) AS avg_profit,
       ROUND(AVG(CASE WHEN "Order Profit Per Order" < 0 THEN 1 ELSE 0 END) * 100, 1) AS loss_pct
FROM orders
GROUP BY discount_pct
ORDER BY discount_pct
"""
print(con.execute(query).df())

    discount_pct  items  avg_profit  loss_pct
0            0.0  10028       26.67      18.4
1            1.0  10028       23.80      18.4
2            2.0  10028       23.67      18.3
3            3.0  10029       22.18      19.2
4            4.0  10029       23.40      18.8
5            5.0  10029       23.78      18.7
6            6.0  10029       24.07      18.1
7            7.0  10029       23.27      18.9
8            9.0  10029       22.91      18.1
9           10.0  10029       22.13      18.4
10          12.0  10029       20.91      19.0
11          13.0  10029       21.38      19.1
12          15.0  10029       20.06      18.8
13          16.0  10029       20.88      18.9
14          17.0  10029       19.33      19.5
15          18.0  10029       20.28      18.4
16          20.0  10029       17.68      19.6
17          25.0  10029       19.13      18.5


In [5]:
query = """
SELECT "Category Name" AS category,
       COUNT(*) AS items,
       ROUND(SUM("Order Profit Per Order"), 0) AS total_profit,
       ROUND(AVG(CASE WHEN "Order Profit Per Order" < 0 THEN 1 ELSE 0 END) * 100, 1) AS loss_pct
FROM orders
GROUP BY category
ORDER BY items DESC
LIMIT 15
"""
print(con.execute(query).df())

                category  items  total_profit  loss_pct
0                 Cleats  24551      494637.0      18.7
1         Men's Footwear  22246      311903.0      18.7
2        Women's Apparel  21035      350421.0      18.6
3   Indoor/Outdoor Games  19298      318451.0      18.7
4                Fishing  17325      756221.0      18.5
5           Water Sports  15540      325147.0      18.8
6       Camping & Hiking  13729      427456.0      18.9
7       Cardio Equipment  12487      383011.0      18.7
8          Shop By Sport  10984      129814.0      19.6
9            Electronics   3156       40891.0      17.8
10           Accessories   1780       16644.0      17.9
11            Golf Balls   1475        8343.0      17.8
12        Girls' Apparel   1201       17289.0      19.3
13           Golf Gloves   1070       12975.0      16.8
14              Trade-In    974        7915.0      18.2


In [6]:
query = """
SELECT "Shipping Mode" AS mode,
       COUNT(*) AS items,
       SUM("Late_delivery_risk") AS late_items,
       ROUND(AVG("Late_delivery_risk") * 100, 1) AS late_pct,
       ROUND(SUM("Late_delivery_risk") * 100.0 / SUM(SUM("Late_delivery_risk")) OVER (), 1) AS share_of_all_late
FROM orders
WHERE "Delivery Status" <> 'Shipping canceled'
GROUP BY mode
ORDER BY late_items DESC
"""
print(con.execute(query).df())

             mode   items  late_items  late_pct  share_of_all_late
0  Standard Class  103153     41023.0      39.8               41.4
1    Second Class   33806     26987.0      79.8               27.3
2     First Class   26513     26513.0     100.0               26.8
3        Same Day    9293      4454.0      47.9                4.5


In [7]:
query = """
SELECT DATE_TRUNC('month', "order date (DateOrders)") AS month,
       COUNT(*) AS items,
       ROUND(AVG("Late_delivery_risk") * 100, 1) AS late_pct
FROM orders
WHERE "Delivery Status" <> 'Shipping canceled'
GROUP BY month
ORDER BY month
"""
print(con.execute(query).df().to_string())

        month  items  late_pct
0  2015-01-01   5114      56.3
1  2015-02-01   4487      57.8
2  2015-03-01   5127      57.3
3  2015-04-01   4915      56.2
4  2015-05-01   5087      58.0
5  2015-06-01   4908      56.6
6  2015-07-01   5124      57.4
7  2015-08-01   5025      58.4
8  2015-09-01   4926      59.2
9  2015-10-01   5073      57.3
10 2015-11-01   5019      56.6
11 2015-12-01   5149      57.0
12 2016-01-01   5105      57.9
13 2016-02-01   4683      56.6
14 2016-03-01   5021      57.9
15 2016-04-01   4880      57.7
16 2016-05-01   5044      58.5
17 2016-06-01   4844      59.3
18 2016-07-01   4994      55.2
19 2016-08-01   5169      57.0
20 2016-09-01   4971      57.1
21 2016-10-01   5206      56.8
22 2016-11-01   4957      57.4
23 2016-12-01   5052      58.5
24 2017-01-01   5033      55.6
25 2017-02-01   4675      57.4
26 2017-03-01   5145      57.4
27 2017-04-01   4980      56.3
28 2017-05-01   5035      56.5
29 2017-06-01   4730      56.0
30 2017-07-01   5052      57.7
31 2017-